# Réduction : diagonalisation


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)  # affichage plus lisible


In [ ]:
def partie_reelle_si_reel(Z):
    # numpy travaille souvent en complexes (affichage "2.+0.j") :
    # si toutes les parties imaginaires sont négligeables, on les supprime
    return Z.real if np.allclose(Z.imag, 0) else Z


# 1. Avec numpy : calcul NUMÉRIQUE (approché)


In [ ]:
# --- 1.a. Trois valeurs propres distinctes -----------------------------------
A=np.array([[7,-4,-6],[5,-3,-5],[5,-4,-4]]) # Regarder la somme par ligne...
print(A)
# Affichage des valeurs propres :
print(partie_reelle_si_reel(np.linalg.eigvals(A)))
# Affichage des éléments propres (brut) :
print(np.linalg.eig(A))
# Regardons notamment la 2è partie :
P=partie_reelle_si_reel(np.linalg.eig(A)[1])
print(P)
# Il s'agit de la matrice (enfin, d'UNE matrice) P telle que P^(-1).A.P soit diagonale
# Sa k-ième COLONNE est un vecteur propre (normé !) associé à la k-ième valeur propre
# On le vérifie :
print(np.linalg.inv(P).dot(A).dot(P))
# Les e-15, e-17... sont en réalité des zéros, ils résultent d'approximations
# dans P et son inverse.
# Si on regarde mieux les colonnes de P, on voit que dans la 1ère colonne, elle
# vaut 4.082...e-1 fois la colonne (2,1,1)
# de même, la 2ème colonne vaut clairement -5.77...e-1 fois la colonne (1,1,1)
# (cf. la somme constante sur chaque ligne de A)
# Enfin, la 3ème colonne vaut 7.07...e-1 fois la colonne (1,0,1)
# Bref, la matrice P peut être choisie égale à P1 définie par
P1=np.array([[2,1,1],[1,1,1],[1,0,1]]).transpose()
# J'ai rentré par ligne puis transposé, c'est plus simple !!!
print(P1)
# Vérifions que l'on diagonalise bien A avec P1 :
print(np.linalg.inv(P1).dot(A).dot(P1))
# Ouf, on est rassuré !
# Si on veut changer l'ordre des valeurs propres, il suffit de changer
# l'ordre des colonnes de P1 (donc des lignes avant transposition).
# Par exemple, pour avoir les vp dans l'ordre croissant -3<1<2 :
P2=np.array([[1,1,1],[1,0,1],[2,1,1]]).transpose()
print(P2)
# Vérifions que l'on diagonalise bien A avec P2 :
print(np.linalg.inv(P2).dot(A).dot(P2))
# Et on a bien l'ordre voulu !


In [ ]:
# --- 1.b. Une valeur propre double (et pourtant diagonalisable) --------------
M=np.array([[12,-2,-6],[0,2,0],[20,-4,-10]])
valeurs_propres, Q = np.linalg.eig(M)
valeurs_propres = partie_reelle_si_reel(valeurs_propres)
Q = partie_reelle_si_reel(Q)
print("\nValeurs propres de M (numpy) :", valeurs_propres)
print("Matrice Q (vecteurs propres en colonnes) :")
print(Q)
D = np.diag(valeurs_propres)
print("Vérification : Q^(-1).M.Q =")
print(np.linalg.inv(Q).dot(M).dot(Q))
print("Égal à D (aux erreurs d'arrondi près) ?",
      np.allclose(np.linalg.inv(Q).dot(M).dot(Q), D))
# Ici, deux colonnes de Q sont associées à la valeur propre 2 :
# elles forment une base du sous-espace propre E_2(M), de dimension 2.


In [ ]:
# --- 1.c. ATTENTION : une matrice NON diagonalisable --------------------------
# (c'est la matrice de l'exemple du cours de valeurs propres -3, 2, 2)
C=np.array([[-3,-5,10],[-10,-10,24],[-5,-6,14]])
vp, R = np.linalg.eig(C)
vp, R = partie_reelle_si_reel(vp), partie_reelle_si_reel(R)
print("\nValeurs propres de C (numpy) :", vp)
print("Matrice R renvoyée :")
print(R)
print("det(R) =", np.linalg.det(R))
# numpy renvoie quand même une matrice R... dont deux colonnes sont
# (presque) égales : det(R) est quasi nul, R n'est pas vraiment inversible !
# On remarque aussi des parties imaginaires parasites (de l'ordre de 1e-8,
# donc non éliminées) : la valeur propre double rend le calcul numérique
# instable. Seul le calcul exact permet de conclure avec certitude...


# 2. Faisons mieux avec sympy : calcul EXACT (formel)


In [ ]:
import sympy as sp
X = sp.symbols('X')


In [ ]:
def elements_propres(Mat):
    # Affichage (lisible) des éléments propres d'une matrice sympy
    sp.pprint(Mat)
    print("Polynôme caractéristique :", sp.factor(Mat.charpoly(X).as_expr()))
    for E in Mat.eigenvects():
        print("Valeur propre :", E[0])
        print("Multiplicité :", E[1])
        print(f"Base du sous-espace propre associé (qui est de dimension {len(E[2])}) : ")
        sp.pprint(E[2])
    print("Diagonalisable ?", Mat.is_diagonalizable())
    print()


In [ ]:
As = sp.Matrix(A)
# Affichage des valeurs propres (dictionnaire {valeur propre : multiplicité}) :
print(As.eigenvals())
# Affichage des éléments propres (brut) :
print(As.eigenvects())
# Affichage plus intéressant :
elements_propres(As)


In [ ]:
Ms = sp.Matrix(M)
elements_propres(Ms)
# Valeur propre double, mais sous-espace propre de dimension 2 : diagonalisable !
# sympy donne directement des matrices P, D telles que P^(-1).M.P = D :
Ps, Ds = Ms.diagonalize()
sp.pprint(Ps)
sp.pprint(Ds)
# Vérification exacte (on doit trouver la matrice nulle) :
sp.pprint(Ps.inv() * Ms * Ps - Ds)


In [ ]:
Cs = sp.Matrix(C)
elements_propres(Cs)
# Valeur propre double, mais sous-espace propre de dimension 1 :
# C n'est pas diagonalisable (ce que numpy ne pouvait pas affirmer).


In [ ]:
# Si la matrice n'est pas diagonalisable, sympy peut tout de même donner des
# matrices P et T telles que P^(-1).B.P = T, avec T triangulaire supérieure
# (c'est la "réduite de Jordan", hors programme) :
# Et pour la matrice C du cours :
P, T = Cs.jordan_form()
sp.pprint(P)
sp.pprint(T)
# On retrouve exactement la matrice triangulaire de l'exemple du cours :
# il n'est pas possible de remplacer le 1 par un 0 !
